# Volatility

The _Volatility_ of a stock gives us a measure on the amount of uncertainty given the changes made over time. Normally, it is computed using the standard deviation over the logaritmic returns.

The daily volatility for a given period of returns $P$ can be defined as:

$\displaystyle \sigma _{\text{P}}=\sigma _{\text{daily}}{\sqrt{P}}$

Therefore, to calculate the volatily for one month, we just have to compute the standard deviation over the returns of that month, having $P$ trading days:

In [1]:
from math import sqrt
import pandas as pd
prices = pd.read_csv("./data/prices.csv", index_col=[0,1], parse_dates=[1])

def stock_prices(ticker):
  return prices.loc[ticker].copy()

msft = stock_prices("msft")

time_slice = msft.Close['2016-03-30':'2016-04-29']
std_dev_returns = time_slice.pct_change().std()
trading_days = len(time_slice)

volatility = std_dev_returns * sqrt(trading_days)
volatility

np.float64(0.08540732054646473)

This gives us notion on how far the price might deviate from the average, in this case a 8.54%.

## Beta

Beta (β) is a measure of the volatility in relation to the market as a whole. It's normally computed against an index like SP&500. Stocks having β values close to 1 indicate that its returns move close to the index. Stocks with betas higher than 1.0 can be interpreted as more volatile than the S&P 500.

It's defined as:

$\displaystyle \beta _{i}={\frac {\mathrm {Cov} (r_{i},r_{m})}{\mathrm {Var} (r_{m})}}$

where $r_{i}$ is the return on an individual stock and $r_{m}$ is the return on the overall market.

To calculate it, first we need to get the data for the S&P500 index, represented as `^GSPC`. You can use a library called [pandas_datareader](https://pydata.github.io/pandas-datareader/remote_data.html), that can read stock data from different sources. Currently there are some problems when reading the data from yahoo finance through pandas reader, so alternativelly you can use [yfinance](https://ranaroussi.github.io/yfinance/) or you can read the csv file in `./data/sp500.csv`.

In [2]:
import yfinance as yf

sp500 = yf.download("^GSPC", start="2018-01-01", end="2022-12-31", auto_adjust=True)
sp500.head(3)

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2018-01-02,2695.810059,2695.889893,2682.360107,2683.729980,3397430000
2018-01-03,2713.060059,2714.370117,2697.770020,2697.850098,3544030000
2018-01-04,2723.989990,2729.290039,2719.070068,2719.310059,3697340000


In [3]:
sp500.columns = sp500.columns.droplevel(1)
sp500.columns.name = None
sp500.head(3)

,Close,High,Low,Open,Volume
Date,,,,,
2018-01-02,2695.810059,2695.889893,2682.360107,2683.729980,3397430000
2018-01-03,2713.060059,2714.370117,2697.770020,2697.850098,3544030000
2018-01-04,2723.989990,2729.290039,2719.070068,2719.310059,3697340000


We will compute the Beta value for 5Y period:

In [4]:
aapl = stock_prices("aapl")
r_i = aapl.Close['2018':'2022'].pct_change()
r_m = sp500.Close['2018':'2022'].pct_change()
beta = r_i.cov(r_m) / r_m.var()
beta

np.float64(1.1973329203676981)